### Imports

In [ ]:
import numpy as np
import math
from scipy.stats import norm, t
from scipy import stats

### Part 1 - Crude Monte Carlo

In [ ]:
n = 100

u = np.random.uniform(0, 1, n)
x = np.exp(u)
theta_hat = np.mean(x)

s = np.std(x, ddof=1)

half_width = t.ppf(0.975, n-1) * s / np.sqrt(n)

lower = theta_hat - half_width
upper = theta_hat + half_width

print("Estimate =", theta_hat)
print("95% CI =", (lower, upper))

Estimate = 1.6659956316303495
95% CI = (np.float64(1.5827091057329508), np.float64(1.7492821575277482))


In [ ]:
R = 1000
estimates = []

for _ in range(R):
    u = np.random.uniform(0, 1, n)
    estimates.append(np.mean(np.exp(u)))

print("Mean estimate =", np.mean(estimates))
print("Estimator variance =", np.var(estimates))

Mean estimate = 1.716599394786136
Estimator variance = 0.0023008648938025204


### Part 2

In [ ]:
n_total_evaluations = 100
n_pairs = n_total_evaluations // 2 

u = np.random.rand(n_pairs)

y1 = np.exp(u)
y2 = np.exp(1 - u)

y = (y1 + y2) / 2

point_estimate = np.mean(y)

sem = stats.sem(y)
ci = stats.t.interval(0.95, df=n_pairs-1, loc=point_estimate, scale=sem)

print(f"Point Estimate: {point_estimate:.6f}")
print(f"95% Confidence Interval: ({ci[0]:.6f}, {ci[1]:.6f})")

Point Estimate: 1.701602
95% Confidence Interval: (1.686140, 1.717064)


### Part 3 - Control Variates

In [ ]:
n = 100

u = np.random.uniform(0, 1, n)

x = np.exp(u)

theta_mc = np.mean(x)

print(theta_mc)

1.7049242150975379


In [ ]:
n = 100

u = np.random.uniform(0, 1, n)

x = np.exp(u)

mu_z = 0.5

cov_xz = np.cov(x, u, ddof=1)[0, 1]

var_z = np.var(u, ddof=1)

c = -cov_xz / var_z

y = x + c * (u - mu_z)

theta_cv = np.mean(y)

print(theta_cv)

1.7278561692416832


In [ ]:
alpha = 0.05

std = np.std(y, ddof=1)

half_width = (
    norm.ppf(0.975)
    * std
    / np.sqrt(n)
)

lower = theta_cv - half_width
upper = theta_cv + half_width

print("Estimate:", theta_cv)
print("95% CI:", (lower, upper))

Estimate: 1.7278561692416832
95% CI: (np.float64(1.714222173312291), np.float64(1.7414901651710752))


In [ ]:
print("MC variance:", np.var(x, ddof=1))
print("CV variance:", np.var(y, ddof=1))

reduction = (
    1
    - np.var(y, ddof=1)
      / np.var(x, ddof=1)
)

print("Variance reduction:", reduction)

MC variance: 0.281927700915916
CV variance: 0.004838938894810042
Variance reduction: 0.9828362417772731


### Part 4

In [ ]:
def CI_mean(means, confidence=0.95):
    n = len(means)
    mean = np.mean(means)
    t_test_statistic = t.ppf(1 - 0.05/2, df=n-1)
    S_n = np.std(means, ddof=1)
    lower_bound = mean - t_test_statistic * S_n / np.sqrt(n)
    upper_bound = mean + t_test_statistic * S_n / np.sqrt(n)
    return mean,lower_bound, upper_bound

In [ ]:
sub_intervals = 10
lower_interval = np.arange(0, 1.0, 1/sub_intervals)
upper_interval = np.arange(0.1, 1.1, 1/sub_intervals)

intervals = np.array([lower_interval, upper_interval]).T

U_stratisfied = np.zeros((n//sub_intervals,sub_intervals))

for j in range(1, sub_intervals+1):
    U_stratisfied[:, j-1] = np.random.uniform(((j-1)/sub_intervals), (j/sub_intervals), n//sub_intervals)

Y_i = np.mean(np.exp(U_stratisfied), axis=1)
theta_SS = np.mean(Y_i)

print("Point estimate:", theta_SS)
print("Confidence Interval of theta MC", CI_mean(Y_i)[1:])
print("Analytical Solution:", np.e -1)
    

Point estimate: 1.7225271793809331
Confidence Interval of theta MC (np.float64(1.7115993633715454), np.float64(1.7334549953903209))
Analytical Solution: 1.718281828459045


### Part 5

In [ ]:
class BlockingSystem():
    def __init__(self, m_service_time=8, m_arriving_time=1, num_servers=10, num_customers=10*10000):
        self.lamd = 1/m_arriving_time # mean inter-arrival time
        self.mu = 1/m_service_time # mean service time
        self.num_servers = num_servers
        self.num_customers = num_customers
        
        self.servers_available = self.num_servers # m
        self.servers_status = self.servers_status = np.zeros(self.num_servers)
        self.blocked = 0
        
    def sample_service_time(self): # ~ Exp(mu)
        U = np.random.rand()
        return -1* np.log(U) /self.mu 

    def sample_arrival_time(self): # ~ Exp(lamd)
        U = np.random.rand()
        return -1 * np.log(U) /self.lamd 
    
    
    def allocate_server(self, arrival_time, service_time):
        servers_available = np.sum(self.servers_status <= arrival_time) # Number of servers that are available at the time of arrival
        
        if servers_available > 0:
            bool = True
            allocated_server = np.argmin(self.servers_status)
            self.servers_status[allocated_server] = arrival_time + service_time
            
        else:
            bool = False
        return bool
    
    def blocking_probability(self):
        return self.blocked / ( self.num_customers)    
    
    def analytical_blocking_probability(self): 
        A = self.lamd / self.mu # lambda * s
        numerator = A**self.num_servers / math.factorial(self.num_servers)
        denominator = np.sum([A**i / math.factorial(i) for i in range(self.num_servers + 1)])
        
        
        return numerator / denominator # B = P(m)
    
    def reset(self):
        self.servers_status = np.zeros(self.num_servers)
        self.blocked = 0
        
    
    def simulate(self):
        t = 0
        c = 0
        self.event_array = np.zeros((self.num_customers, 2))
        arrival_times = np.zeros(self.num_customers)
        self.service_times = np.zeros(self.num_customers)
        
        for c in range(self.num_customers): # Each iteration corresponds to one customer
            
            
            customer = np.zeros(2) #arrival time, departure time
            
            c_arrival_time = self.sample_arrival_time()
            arrival_times[c] = c_arrival_time
            customer[0] = t + c_arrival_time
            service_time = self.sample_service_time()
            

            if self.allocate_server(customer[0], service_time):
                customer[1] = customer[0] + service_time
                self.service_times[c] = service_time
                
            else:
                customer[1] = -1
                self.blocked += 1
                self.service_times[c] = -1
                
            self.event_array[c, :] = customer
            
            
            
            t += c_arrival_time
            
    def plot_A_t(self, ax=None, t_max=10):
        if ax is None:
            ax = plt.gca()
        count = np.arange(1, self.num_customers + 1)
        ax.plot(self.event_array[:, 0], count, label='Arrivals')
        ax.plot(np.sort(self.event_array[:, 1][self.event_array[:, 1] != -1]), count[self.event_array[:, 1] != -1], label='Departures', c='green')
        ax.plot(self.event_array[:, 0][self.event_array[:, 1] == -1], count[:len(self.event_array[:, 0][self.event_array[:, 1] == -1])], label='Rejections', c='red')
        
        
        
        ax.set_xlabel('Time')
        ax.set_ylabel('Event Count')
        ax.set_xlim(0, t_max)
        ax.set_ylim(0, self.event_array[:, 0][self.event_array[:, 0] <= t_max].shape[0]*1.1)
        ax.set_title(f'Events Over Time with $\\mu={self.mu}$, $\\lambda={self.lamd}$, $m={self.num_servers}$')
        ax.legend()
        ax.grid()
        #plt.show()
    
    def plot_inter_times(self, ax=None):
        if ax is None:
            ax = plt.gca()
        inter_arrival_times = np.diff(self.event_array[:, 0])
        inter_departure_times = np.diff(np.sort(self.event_array[:, 1][self.event_array[:, 1] != -1]))
        
        ax.hist(inter_arrival_times, bins=50, density=True, alpha=0.6, color='g', label='Inter-arrival Times')
        ax.hist(inter_departure_times, bins=50, density=True, alpha=0.6, color='b', label='Inter-departure Times')
        
        ax.set_title('Inter-arrival and Inter-departure Times Distribution')
        ax.set_xlabel('Time')
        ax.set_ylabel('Density')
        ax.grid()
        ax.legend()
        plt.show()
            
            

In [ ]:
num_simulations = 100
X_i = np.zeros(num_simulations)
blocking_system = BlockingSystem()

Z_i = np.zeros(num_simulations)

for i in range(num_simulations):
    blocking_system.reset()
    blocking_system.simulate()
    X_i[i] = blocking_system.blocking_probability()
    arrival_times = np.diff(blocking_system.event_array[:, 0])
    Z_i[i] = np.mean(arrival_times)
    
c = -1 *np.cov(X_i, Z_i, ddof=1)[0,1] / np.var(Z_i, ddof=1)
Y_i = X_i + c*(Z_i - 1/blocking_system.lamd)
print(f"c = {c:.3f}")

theta = np.mean(X_i)
var_theta = np.var(X_i, ddof=1)
print(f"Theta hat: {theta:.4f}")
print(f"Theta (analytical probability): {blocking_system.analytical_blocking_probability():.4f}")


theta_SS = np.mean(Y_i)
var_theta_SS = np.var(Y_i, ddof=1)
print(f"Theta_SS hat: {theta_SS:.4f}")

t_test_statistic = t.ppf(1 - 0.05/2, df=num_simulations-1)
CI = (theta - t_test_statistic*np.sqrt(var_theta)/np.sqrt(num_simulations), theta + t_test_statistic*np.sqrt(var_theta)/np.sqrt(num_simulations))

print(f"95% CI Theta hat: [{CI[0]:.4f}, {CI[1]:.4f}]")

t_test_statistic = t.ppf(1 - 0.05/2, df=num_simulations-1)
CI = (theta_SS - t_test_statistic*np.sqrt(var_theta_SS)/np.sqrt(num_simulations), theta_SS + t_test_statistic*np.sqrt(var_theta_SS)/np.sqrt(num_simulations))

print(f"95% CI Theta SS hat: [{CI[0]:.4f}, {CI[1]:.4f}]")



c = 0.335
Theta hat: 0.1215
Theta (analytical probability): 0.1217
Theta_SS hat: 0.1217
95% CI Theta hat: [0.1212, 0.1218]
95% CI Theta SS hat: [0.1214, 0.1219]


In [ ]:
print(f"Variance reduction: {(var_theta_SS - var_theta)*100 / var_theta}%")

Variance reduction: -28.50195016038008%


### Part 7

In [ ]:
def compare_event_estimators():
    a_values = [2.0, 4.0]
    n_samples_list = [10000, 100000]
    sigma2_values = [0.5, 1.0]
    
    print(f"{'a':<5} | {'N':<8} | {'sigma2':<6} | {'Crude MC':<10} | {'Imp. Samp':<10} | {'Efficiency (Var)'}")
    print("-" * 75)

    for a in a_values:
        for n in n_samples_list:
            for s2 in sigma2_values:
                # Crude Monte Carlo (CMC)
                z = np.random.normal(0, 1, n)
                crude_indicator = (z > a)
                p_crude = np.mean(crude_indicator)
                var_crude = np.var(crude_indicator) / n
                
                # Importance Sampling (IS)
                # Sampling from a shifted proposal distribution g(x) ~ N(a, sigma^2)
                x = np.random.normal(a, np.sqrt(s2), n)
                
                # Calculate weights: likelihood ratio f(x)/g(x)
                f_x = norm.pdf(x, 0, 1)
                g_x = norm.pdf(x, a, np.sqrt(s2))
                weights = f_x / g_x
                
                # Weighted estimator
                is_values = (x > a) * weights
                p_is = np.mean(is_values)
                var_is = np.var(is_values) / n
                
                # Efficiency: Ratio of variances (Var_CMC / Var_IS)
                efficiency = var_crude / var_is
                
                print(f"{a:<5} | {n:<8} | {s2:<6} | {p_crude:<10.5f} | {p_is:<10.5f} | {efficiency:.2f}x")


compare_event_estimators()

a     | N        | sigma2 | Crude MC   | Imp. Samp  | Efficiency (Var)
---------------------------------------------------------------------------
2.0   | 10000    | 0.5    | 0.02320    | 0.02251    | 29.23x
2.0   | 10000    | 1.0    | 0.02310    | 0.02253    | 18.43x
2.0   | 100000   | 0.5    | 0.02347    | 0.02269    | 29.53x
2.0   | 100000   | 1.0    | 0.02274    | 0.02268    | 18.30x
4.0   | 10000    | 0.5    | 0.00000    | 0.00003    | 0.00x
4.0   | 10000    | 1.0    | 0.00000    | 0.00003    | 0.00x
4.0   | 100000   | 0.5    | 0.00007    | 0.00003    | 23627.01x
4.0   | 100000   | 1.0    | 0.00003    | 0.00003    | 6616.72x


### Part 8

In [ ]:
def sample_exp(lam, n):
    U = np.random.uniform(size=n)
    Y = -np.log(1 - U * (1 - np.exp(-lam))) / lam
    
    g = lam * np.exp(-lam * Y) / (1 - np.exp(-lam))
    weights = np.exp(Y) / g
    
    return weights.mean(), weights.var(ddof=1)

n = 10000

for lam in [0.5, 1.0, 2.0, 5.0]:
    est, var = sample_exp(lam, n)
    print(f"lambda={lam:.1f}  estimate={est:.6f}  Var={var:.6f}")

print()
print(np.e - 1)

lambda=0.5  estimate=1.719197  Var=0.566705
lambda=1.0  estimate=1.734320  Var=1.091739
lambda=2.0  estimate=1.721409  Var=2.885193
lambda=5.0  estimate=1.695526  Var=28.469250

1.718281828459045


In [ ]:
# part 9

# Pareto has density f(x) = k/beta * (x/beta)^(-k-1) and the first moment distribution has density g(x) = (k-1)/β * (x/β)^(-k)

# The IS weight is f(x) / g(x) = k/(k-1) * beta/x

# So the IS estimate of the mean is:
# E_g[ x * f(x)/g(x) ] = E_g[ x * k/(k-1) * beta/x ]
#                         = E_g[ k*beta/(k-1) ]
#                        = k*beta/(k-1)

#  Which is formula for the Pareto mean => the variance is zero